In [1]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

from dotenv import load_dotenv
load_dotenv()

/Users/apple/Desktop/Artificial Intelligence/GEN AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/apple/Desktop/Artificial Intelligence/GEN AI/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [45]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [46]:
#create a tool 
from langchain_core.tools import InjectedToolArg
from typing import Annotated 
@tool 
def get_conversion_factor(base_currency : str , target_currency : str) -> float:
    """this function fetches the currency conversion factor between a given base currency and target currency"""
    url = f'https://v6.exchangerate-api.com/v6/40b0c9086b8e5b0f7cc5b6aa/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()


@tool 
def convert(base_currency_value : int , convertion_rate : Annotated[float  , InjectedToolArg] ) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a base currency value
    """
    return base_currency_value * convertion_rate

get_conversion_factor({'base_currency': 'USD' , 'target_currency' : 'INR'} )


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782691201,
 'time_last_update_utc': 'Mon, 29 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782777601,
 'time_next_update_utc': 'Tue, 30 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4902}

In [47]:
convert.invoke({'base_currency_value' : 10 , 'convertion_rate':94.4902 })

944.902

In [48]:
model_with_tool =  model.bind_tools([get_conversion_factor , convert])

In [49]:
messages = [HumanMessage("what is the conversion factor between USD and INR  ,based on that can you convert 10 USD to INR ")]

In [50]:
ai_message = model_with_tool.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6y1x6qvx4', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '0we7wfc3k', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 346, 'total_tokens': 384, 'completion_time': 0.112217148, 'completion_tokens_details': None, 'prompt_time': 0.046912866, 'prompt_tokens_details': None, 'queue_time': 0.049699428, 'total_time': 0.159130014}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f11bf-1637-7642-bf0d-f07e4cef6b7c-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '6y1x6qvx4', 'type': 'tool_call'}, {'name': 'convert', 'args': {'bas

In [51]:
messages.append(ai_message)

In [52]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '6y1x6qvx4',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '0we7wfc3k',
  'type': 'tool_call'}]

In [53]:
import json
for tool_call in ai_message.tool_calls:
    #execute the first tool and get the value of convertion rate 
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch the convertion rate 
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        #append this into messages 
        messages.append(tool_message1)

    #execute the second tool sing the convertion rate from tool 1
    if tool_call['name'] == 'convert':
        #fetch the current argument
        tool_call['args']['convertion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)

        messages.append(tool_message2)



In [54]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR  ,based on that can you convert 10 USD to INR ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6y1x6qvx4', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': '0we7wfc3k', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 346, 'total_tokens': 384, 'completion_time': 0.112217148, 'completion_tokens_details': None, 'prompt_time': 0.046912866, 'prompt_tokens_details': None, 'queue_time': 0.049699428, 'total_time': 0.159130014}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f11bf-1637-7642-bf0d-f07e4cef6b7c-0', tool_calls=[

In [56]:
model_with_tool.invoke(messages).content

'The conversion factor between USD and INR is 94.4902. \n10 USD is equivalent to 944.902 INR.'